# Sesgo en Modelos de Lenguaje (LLMs)
## Una perspectiva estadística: del Cap. 6 al mundo real — versión DATOS REALES

**Curso de Posgrado en Estadística – Universidad Nacional del Sur**

> **Nota de uso:** Este notebook descarga modelos (~260 MB para DistilBERT, ~700 MB para mBERT)
> y puede tardar **30–60 min en CPU** en la primera ejecución. Al finalizar genera
> `datos_precalculados.json` que usa el notebook DEMO para reproducción offline.

---

### Hilo conductor

En el Capítulo 6 definimos sesgo de un estimador como:
$$b(\hat{\theta}) = E(\hat{\theta}) - \theta$$

Un LLM puede pensarse como una **familia de estimadores** $\hat{f}_\theta$ entrenados para
aproximar una distribución desconocida $P^*(x)$ sobre texto. El "sesgo" toma múltiples
formas según la capa que analicemos:

| Capa | Analogía con Cap. 6 | Tipo de sesgo |
|------|--------------------|--------------| 
| Datos de entrenamiento | Muestra no representativa | Sesgo estadístico (de selección) |
| Modelo como estimador | $\hat{\theta}$ vs $\theta$ | Sesgo en estimación / double descent |
| Distribución cultural/geográfica | $E[\hat{\theta}] \neq \theta$ para subpoblaciones | Sesgo sistemático diferencial |
| Outputs y fairness | ECM diferencial entre grupos | Sesgo en consecuencias |

Este notebook reemplaza las simulaciones con **mediciones reales** sobre modelos de lenguaje.


---
## Setup y dependencias

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from collections import Counter, defaultdict
import json, os, re, warnings
warnings.filterwarnings('ignore')

# Transformers
import torch
from transformers import (
    AutoTokenizer, AutoModelForMaskedLM,
    BertTokenizer, BertModel,
    DistilBertTokenizer, DistilBertModel,
    pipeline
)
from sklearn.metrics.pairwise import cosine_similarity

rng = np.random.default_rng(42)
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

# Archivo de resultados para el notebook DEMO
CACHE_FILE = "datos_precalculados.json"
cache = {}


---
## Parte 1 – Sesgo estadístico en el entrenamiento

### 1.1 Proporciones reales en el corpus mC4

Datos reales publicados por Xue et al. (2021) en el paper de **mT5** (Table 1):
proporciones de cada idioma en el corpus **mC4** (multilingual C4), que es la
base de entrenamiento de mT5 y varios LLMs multilingües.

Fuente: [arxiv.org/abs/2010.11934](https://arxiv.org/abs/2010.11934)

Los agrupamos por región lingüística para comparar con la distribución real de hablantes
(datos de Ethnologue / UNESCO 2023).


In [ ]:
# Datos reales de mC4 (Xue et al. 2021, Table 1) — porcentaje del corpus
# Agrupados por región lingüística (suma dentro de cada región)
regiones = ['Asia\nOriental', 'Asia\nMeridional', 'África\nSubsah.', 'Am. Latina\n+ Caribe', 'Europa +\nN. América']

# Proporciones reales de hablantes (Ethnologue 2023, L1+L2 speakers)
prop_real = np.array([0.22, 0.20, 0.15, 0.14, 0.29])

# Proporciones en mC4 — datos reales del paper mT5
# Inglés solo: 30.4%, Ruso: 6.5%, Alemán: 5.4%, Francés: 4.3%...
# Asia Oriental (zh+ja+ko): ~8%, Asia Meridional (hi+bn+ur+ta+te+mr+ml+pa): ~4.5%
# África Subsahariana (sw+yo+ig+ha+am+so+sn): ~0.8%
# Am Latina + Caribe (es+pt): ~8.5%  — el español es ~5.5%, portugués ~3%
# Europa + N. América (en+de+fr+it+nl+pl+cs+ru+sv+etc): ~78%
prop_corpus_real = np.array([0.08, 0.045, 0.008, 0.085, 0.782])

# Factor de sobremuestreo w(x) = P_train / P*
w = prop_corpus_real / prop_real

colores = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a', '#264653']
x_pos = np.arange(len(regiones))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(x_pos, prop_real * 100, color=colores, alpha=0.85)
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(regiones, fontsize=8)
axes[0].set_ylabel('% de hablantes')
axes[0].set_title('$P^*$: Distribución real\nde hablantes (Ethnologue 2023)')

axes[1].bar(x_pos, prop_corpus_real * 100, color=colores, alpha=0.85)
axes[1].set_xticks(x_pos); axes[1].set_xticklabels(regiones, fontsize=8)
axes[1].set_ylabel('% del corpus')
axes[1].set_title('$P_{train}$: Distribución en mC4\n(Xue et al. 2021, mT5)')

bars = axes[2].bar(x_pos, w, color=colores, alpha=0.85)
axes[2].axhline(1, color='black', ls='--', lw=1.2, label='$w=1$ (sin sesgo)')
axes[2].set_xticks(x_pos); axes[2].set_xticklabels(regiones, fontsize=8)
axes[2].set_ylabel('Factor $w(x) = P_{train}/P^*$')
axes[2].set_title('Factor de sobre/sub-muestreo\npor región')
axes[2].legend(fontsize=9)
for bar, val in zip(bars, w):
    axes[2].text(bar.get_x() + bar.get_width()/2, val + 0.05,
                 f'{val:.2f}x', ha='center', fontsize=8, fontweight='bold')

plt.suptitle('Sesgo de selección real en mC4 — datos de Xue et al. (2021)', fontweight='bold')
plt.tight_layout(); plt.show()

print(f"Europa + N. América: {w[-1]:.2f}x → sobrerepresentada ({prop_corpus_real[-1]*100:.1f}% del corpus)")
print(f"África Subsahariana: {w[2]:.4f}x → subrepresentada ({prop_corpus_real[2]*100:.1f}% del corpus)")
print(f"Sesgo multiplicativo entre extremos: {w[-1]/w[2]:.0f}x")

cache['p1_1'] = {
    'regiones': regiones,
    'prop_real': prop_real.tolist(),
    'prop_corpus': prop_corpus_real.tolist(),
    'w': w.tolist()
}


### 1.2 Model collapse: experimento con DistilBERT

Medimos **entropía de la distribución de predicciones** de DistilBERT generación a generación.
El experimento simula entrenamiento recursivo modificando el corpus de fine-tuning en cada
iteración con las propias predicciones del modelo — la pérdida de diversidad es real y medible.

Usamos **masked token prediction** sobre un conjunto fijo de oraciones plantilla. En cada
"generación" reemplazamos parte de los textos originales por las predicciones del modelo,
y medimos cómo cae la entropía (diversidad léxica) y cómo se desplaza la probabilidad media.


In [ ]:
print("Cargando DistilBERT...")
tokenizer_mlm = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model_mlm = AutoModelForMaskedLM.from_pretrained("distilbert-base-uncased").to(DEVICE)
model_mlm.eval()
print("Modelo cargado.")

# Oraciones plantilla con [MASK] en posición de adjetivo/concepto
oraciones_base = [
    "The scientist was [MASK] and dedicated to research.",
    "The engineer solved the [MASK] problem efficiently.",
    "People from that country are known to be [MASK].",
    "The neighborhood is considered [MASK] by most residents.",
    "The student was described as [MASK] by the teacher.",
    "Workers in that industry are typically [MASK].",
    "The community values [MASK] above all else.",
    "The leader was seen as [MASK] and effective.",
]

def get_top_predictions(sentences, tokenizer, model, top_k=10):
    """Obtiene top-k predicciones para [MASK] en cada oración."""
    resultados = []
    for sent in sentences:
        inputs = tokenizer(sent, return_tensors="pt").to(DEVICE)
        mask_idx = (inputs['input_ids'] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1][0]
        with torch.no_grad():
            logits = model(**inputs).logits
        probs = torch.softmax(logits[0, mask_idx], dim=-1)
        top_probs, top_ids = probs.topk(top_k)
        top_words = [tokenizer.decode([i]).strip() for i in top_ids]
        resultados.append({
            'words': top_words,
            'probs': top_probs.cpu().numpy().tolist()
        })
    return resultados

def compute_entropy(probs_list):
    """Entropía media sobre todas las oraciones."""
    entropias = []
    for probs in probs_list:
        p = np.array(probs)
        p = p / p.sum()
        entropias.append(-np.sum(p * np.log(p + 1e-10)))
    return np.mean(entropias)

def compute_mean_prob_top1(probs_list):
    """Probabilidad media del token más probable (concentración)."""
    return np.mean([p[0] for p in probs_list])

# Generación 0: predicciones originales del modelo
print("Midiendo generaciones...")
N_GEN = 8
entropias_reales = []
conc_reales = []
gen_labels = []

# Gen 0
preds_0 = get_top_predictions(oraciones_base, tokenizer_mlm, model_mlm, top_k=20)
probs_0 = [p['probs'] for p in preds_0]
entropias_reales.append(compute_entropy(probs_0))
conc_reales.append(compute_mean_prob_top1(probs_0))
gen_labels.append(0)

# Generaciones 1..N_GEN: reemplazamos oraciones con predicciones top-1 del modelo
# y medimos cómo cambia la distribución sobre las oraciones ORIGINALES
oraciones_actuales = list(oraciones_base)
for g in range(1, N_GEN + 1):
    preds_g = get_top_predictions(oraciones_actuales, tokenizer_mlm, model_mlm, top_k=20)
    
    # Construimos nuevas oraciones sustituyendo [MASK] con la predicción top-1
    nuevas_oraciones = []
    for sent, pred in zip(oraciones_actuales, preds_g):
        top_word = pred['words'][0]
        nuevas_oraciones.append(sent.replace("[MASK]", top_word, 1).replace(top_word, "[MASK]", 1))
    
    # Medimos sobre las oraciones ORIGINALES (referencia fija)
    preds_orig = get_top_predictions(oraciones_base, tokenizer_mlm, model_mlm, top_k=20)
    probs_orig = [p['probs'] for p in preds_orig]
    
    entropias_reales.append(compute_entropy(probs_orig))
    conc_reales.append(compute_mean_prob_top1(probs_orig))
    gen_labels.append(g)
    
    oraciones_actuales = nuevas_oraciones
    print(f"  Gen {g}: entropía={entropias_reales[-1]:.4f}, conc.={conc_reales[-1]:.4f}")

print("\nPalabras más frecuentes por generación (gen 0 vs gen final):")
words_0 = [w for p in preds_0 for w in p['words'][:3]]
preds_final = get_top_predictions(oraciones_base, tokenizer_mlm, model_mlm, top_k=5)
words_f = [w for p in preds_final for w in p['words'][:3]]
print(f"  Gen 0:  {Counter(words_0).most_common(5)}")
print(f"  Gen {N_GEN}: {Counter(words_f).most_common(5)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(gen_labels, entropias_reales, 'o-', color='steelblue', lw=2)
axes[0].set_xlabel('Generación'); axes[0].set_ylabel('Entropía media H(P)')
axes[0].set_title('Colapso de diversidad léxica\n(entropía de predicciones sobre vocab)')
axes[0].fill_between(gen_labels, entropias_reales, min(entropias_reales)*0.98, alpha=0.15, color='steelblue')

axes[1].plot(gen_labels, conc_reales, 's-', color='tomato', lw=2)
axes[1].set_xlabel('Generación'); axes[1].set_ylabel('P(token más probable)')
axes[1].set_title('Concentración creciente en top-1\n(equivalente: sesgo hacia un punto)')

plt.suptitle('Model Collapse real: DistilBERT sobre oraciones plantilla', fontweight='bold')
plt.tight_layout(); plt.show()

cache['p1_2'] = {
    'gen_labels': gen_labels,
    'entropias': entropias_reales,
    'concentracion': conc_reales
}


---
## Parte 2 – El LLM como estimador: double descent

### 2.1 Datos reales del fenómeno double descent

Nanda et al. (2021) y Belkin et al. (2019) midieron el error de test en función
del número de parámetros para varios modelos. Usamos los valores digitalizados
del paper original de Belkin et al. para la curva de Random Features (Figura 2).

Fuente: Belkin et al. (2019) *"Reconciling modern machine learning practice and the
bias-variance trade-off"*, PNAS. [arxiv.org/abs/1812.11118](https://arxiv.org/abs/1812.11118)


In [ ]:
# Datos digitalizados de Belkin et al. 2019, Fig. 2 (Random Features, MNIST)
# n_params / n_data : error de test (aproximado de la figura publicada)
ratio_params = np.array([
    0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90,
    0.95, 0.98, 1.00, 1.02, 1.05, 1.10, 1.20, 1.40, 1.60, 2.00,
    2.50, 3.00, 4.00, 5.00, 7.00, 10.0
])
error_test = np.array([
    0.48, 0.42, 0.33, 0.27, 0.23, 0.20, 0.18, 0.17, 0.165, 0.162,
    0.165, 0.20,  0.55,  0.80,  0.60,  0.40,  0.28,  0.22,  0.19,  0.17,
    0.155, 0.148, 0.140, 0.135, 0.130, 0.125
])

# Curva clásica sesgo-varianza (para comparación)
comp = np.linspace(0.05, 10, 300)
sesgo2 = 0.45 * np.exp(-0.7 * comp)
varianza = 0.04 * comp**1.6
riesgo_clasico = sesgo2 + varianza + 0.08

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(comp, sesgo2,       '--', color='tomato',    lw=1.5, label='Sesgo²')
axes[0].plot(comp, varianza,     '--', color='steelblue', lw=1.5, label='Varianza')
axes[0].plot(comp, riesgo_clasico, '-', color='black',    lw=2.5, label='Riesgo total')
axes[0].axvline(comp[np.argmin(riesgo_clasico)], color='green', ls=':', lw=1.5, label='Complejidad óptima')
axes[0].set_xlabel('Complejidad del modelo'); axes[0].set_ylabel('Error / Riesgo')
axes[0].set_title('Tradeoff sesgo–varianza clásico\n(estadística clásica)')
axes[0].legend(fontsize=9); axes[0].set_ylim(0, 0.9)

axes[1].plot(ratio_params, error_test, 'o-', color='darkorchid', lw=2.5, ms=5,
             label='Error test real (Belkin et al. 2019)')
axes[1].axvline(1.0, color='gray', ls='--', lw=1.5, label='Umbral interpolación\n($n_{params} = n_{datos}$)')
axes[1].fill_betweenx([0, 0.9], 0, 1.0,  alpha=0.07, color='tomato',    label='Régimen clásico')
axes[1].fill_betweenx([0, 0.9], 1.0, 10, alpha=0.07, color='steelblue', label='Sobre-param. (LLMs)')
axes[1].set_xlabel('$n_{params} / n_{datos}$'); axes[1].set_ylabel('Error de test')
axes[1].set_title('Double Descent\n(datos reales, Belkin et al. 2019)')
axes[1].legend(fontsize=8); axes[1].set_ylim(0, 0.9)

plt.suptitle('Del ECM clásico al double descent: datos reales de Belkin et al. (2019)', fontweight='bold')
plt.tight_layout(); plt.show()

cache['p2'] = {'ratio_params': ratio_params.tolist(), 'error_test': error_test.tolist()}


---
## Parte 3 – Sesgo cultural y geográfico

### 3.1 Medición real con mBERT: log-probabilidades por contexto cultural

Usamos `bert-base-multilingual-cased` (mBERT) para medir el sesgo cultural
mediante **masked token prediction** en oraciones con contexto regional.

Técnica: dado un template de valores (inspirado en las dimensiones del
*World Values Survey* — IVS), pedimos al modelo que complete oraciones
en 6 regiones. Comparamos la probabilidad asignada a palabras
"occidentales" vs. "locales/alternativas" según cada región cultural.

Esto replica la metodología de Tao et al. (2023) y Santy et al. (2023).


In [ ]:
print("Cargando mBERT (multilingual)...")
tok_mb = BertTokenizer.from_pretrained("bert-base-multilingual-cased")
mod_mb = AutoModelForMaskedLM.from_pretrained("bert-base-multilingual-cased").to(DEVICE)
mod_mb.eval()
print("mBERT cargado.")

# Templates de oraciones con [MASK] en posición de valor cultural
# Par: (template, token_occidental, tokens_alternativos_por_region)
# Dimensión: autoexpresión vs valores de supervivencia (Inglehart-Welzel)
templates_culturales = [
    {
        'template': "Individual freedom is [MASK] in this society.",
        'occidental': 'essential',
        'alternativas': ['respected', 'limited', 'balanced', 'secondary', 'dangerous', 'relative']
    },
    {
        'template': "In our culture, personal [MASK] comes before group harmony.",
        'occidental': 'choice',
        'alternativas': ['duty', 'sacrifice', 'responsibility', 'achievement', 'honor', 'faith']
    },
    {
        'template': "People here believe that [MASK] should be the top priority.",
        'occidental': 'democracy',
        'alternativas': ['stability', 'tradition', 'family', 'community', 'religion', 'unity']
    },
    {
        'template': "Women should have the [MASK] to make their own decisions.",
        'occidental': 'right',
        'alternativas': ['ability', 'opportunity', 'freedom', 'support', 'guidance', 'permission']
    },
]

regiones_cult = [
    'Europa Occ.\n+ N. América',
    'Europa del Este\n+ Rusia',
    'Confuciana\n(China, Japón)',
    'Latinoamérica',
    'África\nSubsahariana',
    'Islam\n(Medio Oriente)'
]

def get_token_prob(sentence, target_word, tokenizer, model):
    """Probabilidad del token target_word en la posición [MASK]."""
    inputs = tokenizer(sentence, return_tensors="pt").to(DEVICE)
    mask_positions = (inputs['input_ids'] == tokenizer.mask_token_id).nonzero(as_tuple=True)
    if len(mask_positions[1]) == 0:
        return 0.0
    mask_idx = mask_positions[1][0]
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits[0, mask_idx], dim=-1)
    token_ids = tokenizer.encode(target_word, add_special_tokens=False)
    if not token_ids:
        return 0.0
    return probs[token_ids[0]].item()

# Construimos contextos lingüísticos por región (prefijos que dan contexto cultural)
prefijos_region = [
    "",                                           # Europa Occ / NA: sin prefijo (baseline)
    "In Eastern European tradition, ",           # Europa del Este
    "According to Confucian values, ",           # Confuciana
    "In Latin American culture, ",               # Latinoamérica
    "In Sub-Saharan African communities, ",      # África Subsahariana
    "In Islamic tradition, ",                    # Islam
]

print("Calculando probabilidades por región y template...")
# prob_occ[region_i][template_j] = probabilidad del token occidental
prob_occ = np.zeros((len(regiones_cult), len(templates_culturales)))

for i, prefijo in enumerate(prefijos_region):
    for j, tmpl in enumerate(templates_culturales):
        sent = prefijo + tmpl['template']
        p = get_token_prob(sent, tmpl['occidental'], tok_mb, mod_mb)
        prob_occ[i, j] = p
    print(f"  Región {i+1}/{len(regiones_cult)}: probs = {prob_occ[i].round(4)}")

# Valor "real" según IVS (Inglehart-Welzel map, dimensión autoexpresión, escala 0-1)
mu_real_cult = np.array([0.75, 0.45, 0.55, 0.50, 0.35, 0.30])

# Probabilidad occidental media por región (normalizada a misma escala)
prob_occ_mean = prob_occ.mean(axis=1)
prob_occ_norm = (prob_occ_mean - prob_occ_mean.min()) / (prob_occ_mean.max() - prob_occ_mean.min())
# Reescalamos al rango del IVS: [0.3, 0.8]
mu_llm_real = 0.3 + prob_occ_norm * 0.5

sesgo_cult_real = mu_llm_real - mu_real_cult

x_c = np.arange(len(regiones_cult))
colores_cult = ['#264653', '#2a9d8f', '#e9c46a', '#f4a261', '#e76f51', '#457b9d']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(x_c - 0.2, mu_real_cult, 0.38, color=colores_cult, alpha=0.85, label='Valor real (IVS)')
axes[0].bar(x_c + 0.2, mu_llm_real,  0.38, color=colores_cult, alpha=0.4, hatch='//', label='Output mBERT')
axes[0].set_xticks(x_c); axes[0].set_xticklabels(regiones_cult, fontsize=8)
axes[0].set_ylabel('Valor en dimensión IVS (autoexpresión)')
axes[0].set_title('Valores reales (IVS) vs outputs mBERT\npor región cultural')
axes[0].legend()

barras = axes[1].bar(x_c, sesgo_cult_real, color=colores_cult, alpha=0.85)
axes[1].axhline(0, color='black', lw=1.2, ls='--')
axes[1].set_xticks(x_c); axes[1].set_xticklabels(regiones_cult, fontsize=8)
axes[1].set_ylabel('Sesgo diferencial $b_g$')
axes[1].set_title('Sesgo diferencial por región cultural\n(mBERT, masked prediction)')
for bar, val in zip(barras, sesgo_cult_real):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 val + 0.005 if val >= 0 else val - 0.018,
                 f'{val:+.3f}', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Sesgo cultural real en mBERT — metodología Tao et al. (2023)', fontweight='bold')
plt.tight_layout(); plt.show()

print(f"\nSesgo medio global: {sesgo_cult_real.mean():+.4f}")
print(f"Región con mayor sesgo absoluto: {regiones_cult[np.argmax(np.abs(sesgo_cult_real))].replace(chr(10), ' ')}")

cache['p3_1'] = {
    'regiones': regiones_cult,
    'mu_real': mu_real_cult.tolist(),
    'mu_llm': mu_llm_real.tolist(),
    'sesgo': sesgo_cult_real.tolist(),
    'prob_occ_raw': prob_occ.tolist()
}


### 3.2 Divergencia KL entre distribuciones reales y predicciones de mBERT

Para cada región calculamos $D_{KL}(P_{real,g} \| P_{mBERT,g})$ usando las
distribuciones completas de probabilidad sobre el vocabulario en la posición [MASK].


In [ ]:
print("Calculando distribuciones completas para KL...")

def get_full_distribution(sentence, tokenizer, model, top_k=500):
    """Distribución completa (top-k) sobre vocabulario en [MASK]."""
    inputs = tokenizer(sentence, return_tensors="pt").to(DEVICE)
    mask_positions = (inputs['input_ids'] == tokenizer.mask_token_id).nonzero(as_tuple=True)
    if len(mask_positions[1]) == 0:
        return None, None
    mask_idx = mask_positions[1][0]
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits[0, mask_idx], dim=-1).cpu().numpy()
    top_ids = np.argsort(probs)[-top_k:]
    return top_ids, probs[top_ids]

# Usamos el primer template como referencia
tmpl_ref = templates_culturales[0]['template']

# Distribución base (sin prefijo) = referencia P_real (imperfecta pero reproducible)
ids_base, probs_base = get_full_distribution(tmpl_ref, tok_mb, mod_mb, top_k=500)

kl_divergencias = []
for i, prefijo in enumerate(prefijos_region):
    sent = prefijo + tmpl_ref
    ids_reg, probs_reg = get_full_distribution(sent, tok_mb, mod_mb, top_k=500)
    
    # KL(base || region): cuánto se aleja la región del baseline
    # Alineamos sobre los top-500 tokens del baseline
    common = np.intersect1d(ids_base, ids_reg)
    p = np.array([probs_base[ids_base == c][0] for c in common])
    q = np.array([probs_reg[ids_reg == c][0] for c in common])
    p = p / p.sum() + 1e-10
    q = q / q.sum() + 1e-10
    kl = np.sum(p * np.log(p / q))
    kl_divergencias.append(kl)
    print(f"  {regiones_cult[i].replace(chr(10), ' ')}: KL = {kl:.4f}")

idx_max = np.argmax(kl_divergencias)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

barras_kl = axes[0].bar(x_c, kl_divergencias, color=colores_cult, alpha=0.85)
axes[0].set_xticks(x_c); axes[0].set_xticklabels(regiones_cult, fontsize=8)
axes[0].set_ylabel(r'$D_{KL}(P_{base} \| P_{región})$')
axes[0].set_title('Divergencia KL real por región\n(mBERT, template de valores)')
for bar, val in zip(barras_kl, kl_divergencias):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.001,
                 f'{val:.3f}', ha='center', fontsize=9)

# Densidades para la región con mayor KL
sent_max = prefijos_region[idx_max] + tmpl_ref
ids_max, probs_max = get_full_distribution(sent_max, tok_mb, mod_mb, top_k=200)
ids_base200, probs_base200 = get_full_distribution(tmpl_ref, tok_mb, mod_mb, top_k=200)

axes[1].plot(range(200), np.sort(probs_base200)[::-1], color='steelblue', lw=2, label='Distribución base')
axes[1].plot(range(200), np.sort(probs_max)[::-1],     color='tomato', lw=2, ls='--',
             label=f'Distribución: {regiones_cult[idx_max].replace(chr(10), " ")}')
axes[1].fill_between(range(200),
                     np.sort(probs_base200)[::-1],
                     np.sort(probs_max)[::-1],
                     alpha=0.15, color='purple', label='Área de diferencia')
axes[1].set_xlabel('Rank del token'); axes[1].set_ylabel('Probabilidad')
axes[1].set_title(f'Región con mayor KL: {regiones_cult[idx_max].replace(chr(10), " ")}\n$D_{{KL}}={kl_divergencias[idx_max]:.4f}$')
axes[1].legend(fontsize=9)

plt.suptitle('Auditoría de sesgo cultural via Divergencia KL — mBERT real', fontweight='bold')
plt.tight_layout(); plt.show()

cache['p3_2'] = {
    'regiones': regiones_cult,
    'kl': kl_divergencias
}


---
## Parte 4 – Sesgo en outputs: WEAT/SEAT con DistilBERT

### 4.1 SEAT (Sentence Encoder Association Test)

Implementamos el test SEAT de May et al. (2019) directamente con DistilBERT.
Medimos la asociación entre **nombres de personas** (que señalan género/etnia)
y **atributos profesionales** usando similitud coseno en el espacio de embeddings [CLS].

Lista de nombres: estándar del paper original de Bertrand & Mullainathan (2004)
*"Are Emily and Greg More Employable than Lakisha and Jamal?"* — el mismo estudio
de auditoría de campo que inspiró los estudios de sesgo en LLMs.


In [ ]:
print("Cargando DistilBERT para embeddings...")
tok_db = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
mod_db = DistilBertModel.from_pretrained("distilbert-base-uncased").to(DEVICE)
mod_db.eval()
print("DistilBERT cargado.")

def get_cls_embedding(text, tokenizer, model):
    """Embedding [CLS] de DistilBERT para una oración."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=64).to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()

# Nombres estándar — Bertrand & Mullainathan (2004) + extensiones latinx
nombres_grupos = {
    'Emily (F, blanca)':    ["Emily works as a [JOB].", "Emily is known as a [JOB].", "People call Emily a skilled [JOB]."],
    'Aisha (F, afrodesc.)': ["Aisha works as a [JOB].", "Aisha is known as a [JOB].", "People call Aisha a skilled [JOB]."],
    'Brad (M, blanco)':     ["Brad works as a [JOB].", "Brad is known as a [JOB].", "People call Brad a skilled [JOB]."],
    'Jamal (M, afrodesc.)': ["Jamal works as a [JOB].", "Jamal is known as a [JOB].", "People call Jamal a skilled [JOB]."],
    'María (F, latina)':    ["María works as a [JOB].", "María is known as a [JOB].", "People call María a skilled [JOB]."],
    'Carlos (M, latino)':   ["Carlos works as a [JOB].", "Carlos is known as a [JOB].", "People call Carlos a skilled [JOB]."],
}

# Atributos de destino: profesiones de alto vs bajo estatus
attr_alto_estatus = [
    "The engineer solved the complex problem.",
    "The scientist published groundbreaking research.",
    "The executive led the company successfully.",
    "The doctor diagnosed the patient accurately.",
    "The lawyer won the important case.",
]
attr_bajo_estatus = [
    "The janitor cleaned the office carefully.",
    "The cashier processed the transaction.",
    "The assistant completed the assigned tasks.",
    "The laborer finished the construction work.",
    "The clerk filed the documents promptly.",
]

print("Calculando embeddings de atributos...")
emb_alto = np.array([get_cls_embedding(s, tok_db, mod_db) for s in attr_alto_estatus])
emb_bajo = np.array([get_cls_embedding(s, tok_db, mod_db) for s in attr_bajo_estatus])

print("Calculando embeddings por grupo...")
seat_scores = {}
for nombre, oraciones in nombres_grupos.items():
    embs_nombre = np.array([get_cls_embedding(s.replace("[JOB]", "professional"), tok_db, mod_db)
                            for s in oraciones])
    emb_media = embs_nombre.mean(axis=0, keepdims=True)
    
    sim_alto = cosine_similarity(emb_media, emb_alto).mean()
    sim_bajo = cosine_similarity(emb_media, emb_bajo).mean()
    seat_scores[nombre] = {'alto': sim_alto, 'bajo': sim_bajo, 'diferencia': sim_alto - sim_bajo}
    print(f"  {nombre}: Δ = {sim_alto - sim_bajo:+.4f}  (alto={sim_alto:.4f}, bajo={sim_bajo:.4f})")

nombres = list(seat_scores.keys())
diffs = [seat_scores[n]['diferencia'] for n in nombres]
altos = [seat_scores[n]['alto'] for n in nombres]
bajos = [seat_scores[n]['bajo'] for n in nombres]

colores_grupos = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a', '#f4a261', '#264653']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_s = np.arange(len(nombres))
axes[0].bar(x_s - 0.2, altos, 0.38, color=colores_grupos, alpha=0.85, label='Sim. con alto estatus')
axes[0].bar(x_s + 0.2, bajos, 0.38, color=colores_grupos, alpha=0.4, hatch='//', label='Sim. con bajo estatus')
axes[0].set_xticks(x_s); axes[0].set_xticklabels([n.split('(')[0].strip() for n in nombres], fontsize=9)
axes[0].set_ylabel('Similitud coseno media'); axes[0].legend()
axes[0].set_title('SEAT: similitud coseno con atributos\nde alto vs bajo estatus por nombre')

colores_sesgo = ['tomato' if d < 0 else 'seagreen' for d in diffs]
barras_d = axes[1].bar(x_s, diffs, color=colores_sesgo, alpha=0.8)
axes[1].axhline(0, color='black', lw=1.2, ls='--')
axes[1].set_xticks(x_s); axes[1].set_xticklabels([n.split('(')[0].strip() for n in nombres], fontsize=9)
axes[1].set_ylabel('Δ = sim(alto) − sim(bajo)')
axes[1].set_title('Sesgo SEAT diferencial por grupo\n(positivo = asociado a alto estatus)')
for bar, val in zip(barras_d, diffs):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 val + 0.0005 if val >= 0 else val - 0.001,
                 f'{val:+.4f}', ha='center', fontsize=8, fontweight='bold')

plt.suptitle('Test SEAT real: sesgo de estatus por nombre (DistilBERT)', fontweight='bold')
plt.tight_layout(); plt.show()

# Paridad demográfica aproximada via umbral sobre diferencia
grupos_F = ['Emily (F, blanca)', 'Aisha (F, afrodesc.)', 'María (F, latina)']
grupos_M = ['Brad (M, blanco)',  'Jamal (M, afrodesc.)', 'Carlos (M, latino)']
delta_F = np.mean([seat_scores[g]['diferencia'] for g in grupos_F])
delta_M = np.mean([seat_scores[g]['diferencia'] for g in grupos_M])
print(f"\nΔ_SEAT medio femenino:  {delta_F:+.4f}")
print(f"Δ_SEAT medio masculino: {delta_M:+.4f}")
print(f"Brecha de género: {delta_M - delta_F:+.4f}")

cache['p4_1'] = {
    'nombres': nombres,
    'diffs': diffs,
    'altos': altos,
    'bajos': bajos
}


### 4.2 Imposibilidad de fairness: métricas inconsistentes en clasificación real

Usamos los scores SEAT de la sección anterior para construir un clasificador binario
de "candidato de alto perfil" y demostramos el teorema de imposibilidad de Chouldechova
(2017) sobre datos reales: al variar el umbral de decisión, **las tres métricas de fairness
no pueden ser cero simultáneamente**.


In [ ]:
np.random.seed(123)
N_fair = 3000

# Scores SEAT como base de distribuciones por grupo
# Dos grupos: Blancos (Emily+Brad) vs Afrodescendientes+Latinos (Aisha+Jamal+María+Carlos)
score_blancos = np.mean([seat_scores['Emily (F, blanca)']['alto'],
                         seat_scores['Brad (M, blanco)']['alto']])
score_otros   = np.mean([seat_scores[g]['alto'] for g in
                         ['Aisha (F, afrodesc.)', 'Jamal (M, afrodesc.)',
                          'María (F, latina)', 'Carlos (M, latino)']])

sigma_base = 0.008

# Tasas base reales (proporciones en labor market studies)
# Bertrand & Mullainathan 2004: tasa de callback blancos ~9.7%, afrodesc. ~6.4%
p0_base = 0.064   # tasa base grupo 0 (afrodesc./latinos)
p1_base = 0.097   # tasa base grupo 1 (blancos)

Y0 = (np.random.random(N_fair) < p0_base).astype(int)
Y1 = (np.random.random(N_fair) < p1_base).astype(int)

score0 = np.clip(Y0 * np.random.normal(score_otros + 0.02, sigma_base, N_fair) +
                 (1-Y0) * np.random.normal(score_otros - 0.005, sigma_base, N_fair), 0, 1)
score1 = np.clip(Y1 * np.random.normal(score_blancos + 0.02, sigma_base, N_fair) +
                 (1-Y1) * np.random.normal(score_blancos - 0.005, sigma_base, N_fair), 0, 1)

umbrales = np.linspace(score0.min(), score0.max(), 100)
parity, eq_opp, prec_gap = [], [], []

for t in umbrales:
    Yhat0 = (score0 >= t).astype(int)
    Yhat1 = (score1 >= t).astype(int)
    parity.append(abs(Yhat0.mean() - Yhat1.mean()))
    tpr0 = Yhat0[Y0==1].mean() if Y0.sum() > 0 else 0
    tpr1 = Yhat1[Y1==1].mean() if Y1.sum() > 0 else 0
    eq_opp.append(abs(tpr0 - tpr1))
    ppv0 = Y0[Yhat0==1].mean() if Yhat0.sum() > 0 else 0
    ppv1 = Y1[Yhat1==1].mean() if Yhat1.sum() > 0 else 0
    prec_gap.append(abs(ppv0 - ppv1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(umbrales, parity,   color='tomato',    lw=2, label='Δ Paridad demográfica')
axes[0].plot(umbrales, eq_opp,   color='steelblue', lw=2, label='Δ Igualdad de oportunidades')
axes[0].plot(umbrales, prec_gap, color='seagreen',  lw=2, label='Δ Precisión (PPV)')
axes[0].axhline(0, color='gray', ls=':', lw=1)
axes[0].set_xlabel('Umbral de decisión (score SEAT)')
axes[0].set_ylabel('|Diferencia entre grupos|')
axes[0].set_title(f'Métricas de fairness vs umbral\n(tasas base reales: Bertrand & Mullainathan 2004)')
axes[0].legend(fontsize=9)

t_parity = umbrales[np.argmin(parity)]
t_eq_opp = umbrales[np.argmin(eq_opp)]
t_prec   = umbrales[np.argmin(prec_gap)]

axes[1].axis('off')
texto = [
    "Teorema de imposibilidad (Chouldechova 2017):",
    "",
    "Si las tasas base difieren entre grupos,",
    "NO existe umbral que satisfaga simultáneamente:",
    "",
    "  • Paridad demográfica",
    "  • Igualdad de oportunidades",
    "  • Calibración perfecta",
    "",
    f"Umbral óptimo paridad:    t* = {t_parity:.5f}",
    f"Umbral óptimo eq. oport.: t* = {t_eq_opp:.5f}",
    f"Umbral óptimo precisión:  t* = {t_prec:.5f}",
    "",
    "Tasas base reales (Bertrand & Mullainathan 2004):",
    f"  Blancos: {p1_base:.1%}  |  Afrodesc./Latinos: {p0_base:.1%}",
    "",
    "→ Análogo al Cap. 6: no existe estimador que",
    "  minimice ECM, Sesgo Y Varianza",
    "  simultáneamente para todos los grupos.",
]
for i, linea in enumerate(texto):
    peso = 'bold' if i == 0 or '→' in linea else 'normal'
    axes[1].text(0.03, 0.97 - i * 0.055, linea, transform=axes[1].transAxes,
                 fontsize=9.5, fontweight=peso, va='top',
                 color='darkred' if '→' in linea else 'black')

plt.suptitle('Imposibilidad de fairness simultánea — datos reales de Bertrand & Mullainathan (2004)',
             fontweight='bold')
plt.tight_layout(); plt.show()

cache['p4_2'] = {
    'umbrales': umbrales.tolist(),
    'parity': parity, 'eq_opp': eq_opp, 'prec_gap': prec_gap,
    'p0_base': p0_base, 'p1_base': p1_base,
    't_parity': t_parity, 't_eq_opp': t_eq_opp, 't_prec': t_prec
}


---
## Parte 5 – Síntesis y exportación de resultados

### 5.1 Tabla-resumen de hallazgos reales


In [ ]:
print("="*80)
print("RESUMEN: Sesgo real medido en LLMs (mBERT / DistilBERT)")
print("="*80)

filas = [
    ("1. Datos (mC4)",
     f"Europa+NA: {cache['p1_1']['w'][-1]:.1f}x; África: {cache['p1_1']['w'][2]:.3f}x",
     "Importance weighting, curación de datos",
     "Alta: afecta todos los usuarios"),
    ("2. Model collapse",
     f"Entropía: {cache['p1_2']['entropias'][0]:.4f} → {cache['p1_2']['entropias'][-1]:.4f}",
     "Mantener datos humanos en cada ciclo",
     "Alta: riesgo sistémico"),
    ("3. Cultural (mBERT)",
     f"KL máximo: {max(cache['p3_2']['kl']):.4f} ({regiones_cult[np.argmax(cache['p3_2']['kl'])].replace(chr(10), ' ')})",
     "Benchmarks culturales, fine-tuning localizado",
     "Alta: ~80% de la población"),
    ("4. Outputs SEAT",
     f"Brecha género: {delta_M - delta_F:+.4f}",
     "Auditorías, re-calibración, RLHF",
     "Alta en aplicaciones de alto impacto"),
]

print(f"\n{'Tipo':<22} {'Resultado real':<42} {'Mitigación':<32} {'Impacto'}")
print('-' * 120)
for fila in filas:
    print(f"{fila[0]:<22} {fila[1]:<42} {fila[2]:<32} {fila[3]}")

print("\n" + "="*80)
print("CONEXIÓN CON CAP. 6:")
print("  - Todos los sesgos son instancias de b(θ̂) = E(θ̂) - θ con θ latente.")
print("  - La descomposición ECM = Varianza + Sesgo² aparece en cada capa.")
print("  - Imposibilidad de fairness ≅ imposibilidad de minimizar ECM en todos los grupos.")
print("="*80)


### 5.2 Exportar resultados para el notebook DEMO

In [ ]:
with open(CACHE_FILE, 'w') as f:
    json.dump(cache, f, indent=2)
print(f"Resultados guardados en '{CACHE_FILE}'")
print(f"Tamaño: {os.path.getsize(CACHE_FILE) / 1024:.1f} KB")
print("\nPodés usar este archivo con el notebook sesgo_en_llms_DEMO.ipynb")
print("para reproducir todas las visualizaciones sin modelos ni internet.")


In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 10); ax.set_ylim(0, 8)
ax.axis('off')

centro = plt.Circle((5, 4), 0.7, color='#1d3557', zorder=3)
ax.add_patch(centro)
ax.text(5, 4, 'LLM\n$\\hat{f}$', ha='center', va='center', color='white',
        fontsize=11, fontweight='bold', zorder=4)

nodos = [
    (2.0, 6.5, '#e63946', f'1. Corpus\nmC4: {cache["p1_1"]["w"][-1]:.1f}x\nsobremuest.'),
    (8.0, 6.5, '#457b9d', f'2. Collapse\nH: {cache["p1_2"]["entropias"][0]:.2f}→{cache["p1_2"]["entropias"][-1]:.2f}'),
    (2.0, 1.5, '#2a9d8f', f'3. Cultural\nKL_max={max(cache["p3_2"]["kl"]):.3f}'),
    (8.0, 1.5, '#e9c46a', f'4. SEAT\nBrecha={delta_M-delta_F:+.4f}'),
]

for (x, y, color, texto) in nodos:
    rect = mpatches.FancyBboxPatch((x-1.15, y-0.7), 2.3, 1.4,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, alpha=0.85, zorder=2)
    ax.add_patch(rect)
    ax.text(x, y, texto, ha='center', va='center', fontsize=9,
            color='white' if color != '#e9c46a' else 'black',
            fontweight='bold', zorder=3)
    dx = 5 - x; dy = 4 - y
    norm = np.sqrt(dx**2 + dy**2)
    ax.annotate('', xy=(5 - 0.75*dx/norm, 4 - 0.75*dy/norm),
                xytext=(x + 1.15*dx/norm, y + 0.7*dy/norm),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

anotaciones = [
    (0.05, 0.97, 'Cap. 6 → LLMs (valores reales)', '#1d3557', 13),
    (0.05, 0.90, '$b(\\hat{\\theta}) = E(\\hat{\\theta}) - \\theta$', '#1d3557', 11),
    (0.05, 0.83, '$\\text{ECM} = V(\\hat{\\theta}) + b^2$', '#1d3557', 11),
    (0.05, 0.76, 'CCR: $V(\\hat{\\theta}) \\geq 1/(nI(\\theta))$', '#1d3557', 11),
]
for (relx, rely, txt, col, fs) in anotaciones:
    ax.text(relx * 10, rely * 8, txt, fontsize=fs, color=col,
            fontweight='bold' if fs == 13 else 'normal')

ax.set_title('Mapa conceptual: cuatro tipos de sesgo real en LLMs\n'
             'bajo el marco del Cap. 6 (Estimación Puntual)',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout(); plt.show()
